# Exploration des questions GAIA
Récupère les questions depuis le serveur du cours, affiche les métadonnées et prévisualise les fichiers associés.

In [2]:
import os, sys
from io import BytesIO
from pathlib import Path
import requests
import pandas as pd
from dotenv import load_dotenv
from IPython.display import display, HTML, Image, Audio

load_dotenv()

API_BASE = os.getenv("SCORING_API_URL", "https://agents-course-unit4-scoring.hf.space")
print(f"API : {API_BASE}")

API : https://agents-course-unit4-scoring.hf.space


## 1. Téléchargement des questions

In [3]:
resp = requests.get(f"{API_BASE}/questions", timeout=30)
resp.raise_for_status()
questions: list[dict] = resp.json()
print(f"{len(questions)} questions récupérées")

20 questions récupérées


## 2. Vue d'ensemble

In [5]:
df = pd.DataFrame([
    {
        "#": i + 1,
        "task_id": q.get("task_id", ""),
        "level": q.get("Level", q.get("level", "")),
        "file": q.get("file_name", "") or "",
        "question": (q.get("question", "") or "")[:1020],
    }
    for i, q in enumerate(questions)
])

pd.set_option("display.max_colwidth", 1000)
display(df["question"])

0                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                          How many studio albums were published by Mercedes Sosa between 2000 and 2009 (included)? You can use the latest 2022 version of english wikip

In [4]:
with_file = df[df["file"] != ""]
print(f"Questions avec fichier joint : {len(with_file)} / {len(df)}")
ext_counts = with_file["file"].str.rsplit(".", n=1).str[-1].value_counts()
print("\nExtensions :")
print(ext_counts.to_string())

Questions avec fichier joint : 5 / 20

Extensions :
file
mp3     2
png     1
py      1
xlsx    1


## 3. Lecture détaillée d'une question
Modifier `IDX` (0-indexé) pour naviguer entre les questions.

In [5]:
IDX = 9  # ← modifier ici

item = questions[IDX]
task_id  = item.get("task_id", "")
file_name = (item.get("file_name") or "").strip()
level    = item.get("Level", item.get("level", "?"))
question = item.get("question", "")

display(HTML(f"""
<table style='border-collapse:collapse; width:100%'>
  <tr><th style='text-align:left;padding:4px 8px;background:#f0f0f0'>Champ</th>
      <th style='text-align:left;padding:4px 8px;background:#f0f0f0'>Valeur</th></tr>
  <tr><td style='padding:4px 8px'><b>#</b></td><td style='padding:4px 8px'>{IDX + 1}</td></tr>
  <tr><td style='padding:4px 8px'><b>task_id</b></td><td style='padding:4px 8px;font-family:monospace'>{task_id}</td></tr>
  <tr><td style='padding:4px 8px'><b>level</b></td><td style='padding:4px 8px'>{level}</td></tr>
  <tr><td style='padding:4px 8px'><b>file_name</b></td><td style='padding:4px 8px'>{file_name or '(aucun)'}</td></tr>
  <tr><td style='padding:4px 8px;vertical-align:top'><b>question</b></td>
      <td style='padding:4px 8px;white-space:pre-wrap'>{question}</td></tr>
</table>
"""))

Champ,Valeur
#,10
task_id,99c9cc74-fdc8-46c6-8f8d-3ce2d3bfeea3
level,1
file_name,99c9cc74-fdc8-46c6-8f8d-3ce2d3bfeea3.mp3
question,"Hi, I'm making a pie but I could use some help with my shopping list. I have everything I need for the crust, but I'm not sure about the filling. I got the recipe from my friend Aditi, but she left it as a voice memo and the speaker on my phone is buzzing so I can't quite make out what she's saying. Could you please listen to the recipe and list all of the ingredients that my friend described? I only want the ingredients for the filling, as I have everything I need to make my favorite pie crust. I've attached the recipe as Strawberry pie.mp3. In your response, please only list the ingredients, not any measurements. So if the recipe calls for ""a pinch of salt"" or ""two cups of ripe strawberries"" the ingredients on the list would be ""salt"" and ""ripe strawberries"". Please format your response as a comma separated list of ingredients. Also, please alphabetize the ingredients."


## 4. Prévisualisation du fichier associé

In [6]:
LOCAL_FILES_DIR = Path("gaia_files")


def fetch_bytes(task_id: str, file_name: str | None = None) -> bytes | None:
    # 1) Essai via API du cours
    url = f"{API_BASE}/files/{task_id}"
    try:
        r = requests.get(url, timeout=60)
        if r.status_code == 200:
            return r.content
        print(f"HTTP {r.status_code} pour {url}")
    except requests.RequestException as e:
        print(f"Erreur réseau : {e}")

    # 2) Fallback local dans gaia_files/
    candidates: list[Path] = []
    if file_name:
        candidates.append(LOCAL_FILES_DIR / file_name)
        candidates.append(LOCAL_FILES_DIR / f"{task_id[:8]}_{file_name}")

    for p in candidates:
        if p.exists() and p.is_file():
            print(f"Fallback local utilisé : {p}")
            return p.read_bytes()

    return None


def preview_file(blob: bytes, file_name: str) -> None:
    ext = file_name.lower().rsplit(".", maxsplit=1)[-1] if "." in file_name else ""
    print(f"Taille : {len(blob):,} octets  |  Extension : {ext or '?'}")

    if ext == "py":
        src = blob.decode("utf-8", errors="replace")
        display(HTML(f"<pre style='background:#f8f8f8;padding:10px;border-radius:4px;overflow:auto'>{src}</pre>"))

    elif ext == "xlsx":
        dfs = pd.read_excel(BytesIO(blob), sheet_name=None)
        for sheet, df_s in dfs.items():
            print(f"\n--- Feuille : {sheet} ({df_s.shape[0]} lignes × {df_s.shape[1]} cols) ---")
            display(df_s.head(20))

    elif ext == "csv":
        df_c = pd.read_csv(BytesIO(blob))
        print(f"{df_c.shape[0]} lignes × {df_c.shape[1]} cols")
        display(df_c.head(20))

    elif ext in ("png", "jpg", "jpeg"):
        display(Image(data=blob))

    elif ext in ("mp3", "wav", "m4a", "ogg"):
        display(Audio(data=blob, autoplay=False))

    elif ext == "txt":
        txt = blob.decode("utf-8", errors="replace")
        display(HTML(f"<pre style='background:#f8f8f8;padding:10px;border-radius:4px;overflow:auto'>{txt[:5000]}</pre>"))

    elif ext == "json":
        import json
        try:
            data = json.loads(blob)
            pretty = json.dumps(data, indent=2, ensure_ascii=False)[:5000]
            display(HTML(f"<pre style='background:#f8f8f8;padding:10px;border-radius:4px;overflow:auto'>{pretty}</pre>"))
        except Exception as e:
            print(f"JSON invalide : {e}")

    else:
        print(f"Format '{ext}' non prévisualisé — téléchargez pour inspecter.")


if file_name:
    blob = fetch_bytes(task_id, file_name)
    if blob:
        preview_file(blob, file_name)
    else:
        print("Fichier introuvable ni via API ni dans gaia_files/.")
else:
    print("Pas de fichier pour cette question.")

HTTP 404 pour https://agents-course-unit4-scoring.hf.space/files/99c9cc74-fdc8-46c6-8f8d-3ce2d3bfeea3
Fallback local utilisé : gaia_files/99c9cc74-fdc8-46c6-8f8d-3ce2d3bfeea3.mp3
Taille : 179,304 octets  |  Extension : mp3


---
## 🔬 Run agent sur une question précise avec trace des étapes

In [7]:
import sys, os
sys.path.insert(0, os.path.abspath("."))
from dotenv import load_dotenv
load_dotenv(override=True)

from agent import build_graph
from regexs import extract_last_ai_text, normalize_gaia_answer
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage
from IPython.display import display, HTML

QUESTION = (
    "What is the first name of the only Malko Competition recipient from the 20th Century "
    "(after 1977) whose nationality on record is a country that no longer exists?"
)

print("Construction du graphe…")
graph = build_graph()
print("Graphe prêt.")

Construction du graphe…
Graphe prêt.


In [8]:
# LANCEMENT A LAMANO

 
#limit = int(os.getenv("LANGGRAPH_RECURSION_LIMIT", "80"))
#result = graph.invoke(
#    {"messages": [HumanMessage(content=QUESTION)]},
#    config={"recursion_limit": limit},
)
#
#messages = result["messages"]
#print(f"{len(messages)} messages dans la trace.\n")
#for i, msg in enumerate(messages, 1):
#    print(msg)

SyntaxError: unmatched ')' (3827879494.py, line 8)

---
## 7. Exécution pas-à-pas sur une question précise
Lance l'agent sur la question ci-dessous et affiche chaque étape (raisonnement, appel d'outil, réponse d'outil) en temps réel via le streaming LangGraph.

In [12]:
import os, sys, textwrap, time
from dotenv import load_dotenv
from IPython.display import display, HTML
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage

# Assure-toi que le dossier du projet est dans le path
sys.path.insert(0, os.path.dirname(os.path.abspath("agent.py")))
load_dotenv()

from agent import build_graph
from question_context import enrich_question

QUESTION = (#"Combien font 155456877 fois 3516546846 ?"
    "How many studio albums were published by Mercedes Sosa between 2000 and 2009 (included)? You can use the latest 2022 version of english wikipedia."
)
TASK_ID   = None   # pas de task_id connu pour une question manuelle
FILE_NAME = None   # pas de fichier joint

API_BASE = os.getenv("SCORING_API_URL", "https://agents-course-unit4-scoring.hf.space")

enriched = enrich_question(QUESTION, task_id=TASK_ID, file_name=FILE_NAME, api_base=API_BASE)
print("Question envoyée à l'agent :")
print(textwrap.fill(enriched, 100))

Question envoyée à l'agent :
How many studio albums were published by Mercedes Sosa between 2000 and 2009 (included)? You can use
the latest 2022 version of english wikipedia.


In [13]:
# ── Helpers d'affichage ──────────────────────────────────────────────────────
COLORS = {
    "ai":   ("#1a73e8", "#e8f0fe"),
    "tool_call": ("#e65100", "#fff3e0"),
    "tool_result": ("#2e7d32", "#e8f5e9"),
    "final": ("#6a1b9a", "#f3e5f5"),
}

def html_box(title: str, body: str, kind: str) -> str:
    fg, bg = COLORS.get(kind, ("#333", "#fafafa"))
    safe_body = body.replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")
    return (
        f"<div style='margin:6px 0;border-left:4px solid {fg};"
        f"background:{bg};padding:8px 12px;border-radius:4px;font-family:monospace'>"
        f"<b style='color:{fg}'>{title}</b><br>"
        f"<pre style='margin:4px 0 0 0;white-space:pre-wrap;font-size:0.88em'>{safe_body}</pre></div>"
    )

def summarise_content(content) -> str:
    """Extrait le texte d'un message (str ou liste de blocs)."""
    if isinstance(content, str):
        return content
    if isinstance(content, list):
        parts = []
        for block in content:
            if isinstance(block, dict):
                parts.append(block.get("text") or block.get("content") or repr(block))
            else:
                parts.append(str(block))
        return "\n".join(parts)
    return repr(content)

In [16]:
graph = build_graph()

messages = [HumanMessage(content=enriched)]
config   = {"recursion_limit": int(os.getenv("LANGGRAPH_RECURSION_LIMIT", "10"))}

step = 0
seen_ids: set = set()
t0 = time.time()

print("═" * 70)
print("DÉMARRAGE DU RUN")
print("═" * 70)

for chunk in graph.stream({"messages": messages}, config=config, stream_mode="updates"):
    for node_name, node_output in chunk.items():
        node_msgs = node_output.get("messages", [])
        if not isinstance(node_msgs, list):
            node_msgs = [node_msgs]

        for msg in node_msgs:
            mid = getattr(msg, "id", None)
            if mid and mid in seen_ids:
                continue
            if mid:
                seen_ids.add(mid)

            elapsed = f"+{time.time() - t0:.1f}s"

            if isinstance(msg, AIMessage):
                text = summarise_content(msg.content)
                tool_calls = getattr(msg, "tool_calls", []) or []

                if tool_calls:
                    step += 1
                    for tc in tool_calls:
                        tc_name = tc.get("name", "?")
                        tc_args = tc.get("args", {})
                        display(HTML(html_box(
                            f"Étape {step} [{elapsed}]  🔧 Appel outil : {tc_name}",
                            str(tc_args),
                            "tool_call",
                        )))
                elif text.strip():
                    is_final = "final answer" in text.lower()
                    kind = "final" if is_final else "ai"
                    step += 1
                    display(HTML(html_box(
                        f"Étape {step} [{elapsed}]  {'🎯 Réponse finale' if is_final else '💬 Raisonnement'}",
                        text,
                        kind,
                    )))

            elif isinstance(msg, ToolMessage):
                step += 1
                tool_name = getattr(msg, "name", "outil")
                body = summarise_content(msg.content)
                preview = body[:1200] + ("\n…[tronqué]" if len(body) > 1200 else "")
                display(HTML(html_box(
                    f"Étape {step} [{elapsed}]  📥 Résultat : {tool_name}",
                    preview,
                    "tool_result",
                )))

print("═" * 70)
print(f"RUN TERMINÉ  ({time.time() - t0:.1f}s)")
print("═" * 70)

══════════════════════════════════════════════════════════════════════
DÉMARRAGE DU RUN
══════════════════════════════════════════════════════════════════════


══════════════════════════════════════════════════════════════════════
RUN TERMINÉ  (104.3s)
══════════════════════════════════════════════════════════════════════
